# MicroScore credit-risk analysis

This notebook is now a thin research layer over the reusable code in `src/`.
The production-ish logic lives in `microscore.features` and `microscore.modeling`.

In [1]:
from pathlib import Path
import sys

def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists() and (path / "src").exists():
            return path
    return start.parent if start.name == "notebooks" else start


PROJECT_ROOT = find_project_root(Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pandas as pd

from microscore.audit import run_audit
from microscore.features import DEFAULT_DROP_COLUMNS, make_model_frame
from microscore.modeling import results_table, run_experiment
from microscore.paths import resolve_data_path

## Load and inspect data

In [2]:
data_path = resolve_data_path()
print(data_path)
df = pd.read_csv(data_path)

df.head()

C:\Users\Sasha\Desktop\MicroScore\data\raw\credit_risk_dataset.csv


,customer_id,age,gender,employment_status,annual_income,account_age_months,avg_monthly_balance,num_deposits_per_month,avg_deposit_amount,debit_card_usage_frequency,...,online_transfer_frequency,atm_withdrawal_frequency,credit_score,num_open_loans,total_outstanding_debt,late_payment_count,loan_default_history,fraud_flag,loan_application_amount,credit_risk
0,CUST_00000,58,Male,Unemployed,58137.751192,115,524.679407,1,171.342369,49,...,16,1,539,5,14435.423445,2,0,0,15510.576882,1
1,CUST_00001,48,Male,Self-Employed,26174.922827,32,2635.203357,1,985.607164,1,...,1,2,494,5,11263.099341,2,0,0,14819.436498,1
2,CUST_00002,34,Other,Unemployed,75566.837265,14,2334.341061,9,994.310119,42,...,5,6,437,3,15017.144132,5,0,0,10909.806507,1
3,CUST_00003,62,Male,Self-Employed,35197.961516,179,2425.384332,10,366.115346,4,...,7,7,809,1,12626.138476,2,0,0,500.000000,0
4,CUST_00004,27,Female,Self-Employed,12136.998349,225,10.000000,8,786.752258,16,...,16,3,522,1,13576.704320,4,0,0,9514.591618,1


In [3]:
df.info()
df["credit_risk"].value_counts(normalize=True).rename("share")

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 22 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   customer_id                 5000 non-null   str    
 1   age                         5000 non-null   int64  
 2   gender                      5000 non-null   str    
 3   employment_status           5000 non-null   str    
 4   annual_income               5000 non-null   float64
 5   account_age_months          5000 non-null   int64  
 6   avg_monthly_balance         5000 non-null   float64
 7   num_deposits_per_month      5000 non-null   int64  
 8   avg_deposit_amount          5000 non-null   float64
 9   debit_card_usage_frequency  5000 non-null   int64  
 10  debit_card_spending         5000 non-null   float64
 11  mobile_banking_logins       5000 non-null   int64  
 12  online_transfer_frequency   5000 non-null   int64  
 13  atm_withdrawal_frequency    5000 non-null   

credit_risk
1    0.7658
0    0.2342
Name: share, dtype: float64

## Build model frame

The current experiment drops identifier, traditional-score, and target-like columns before training.

In [4]:
X, y = make_model_frame(df)

print("Dropped columns:", DEFAULT_DROP_COLUMNS)
print("Feature matrix shape:", X.shape)
X.head()

Dropped columns: ('customer_id', 'credit_score', 'loan_default_history', 'fraud_flag')
Feature matrix shape: (5000, 23)


,age,gender,employment_status,annual_income,account_age_months,avg_monthly_balance,num_deposits_per_month,avg_deposit_amount,debit_card_usage_frequency,debit_card_spending,...,num_open_loans,total_outstanding_debt,late_payment_count,loan_application_amount,income_to_debt_ratio,digital_activity_score,deposit_to_spending_ratio,loan_to_income_ratio,total_credit_pressure,debt_per_open_loan
0,58,Male,Unemployed,58137.751192,115,524.679407,1,171.342369,49,768.877511,...,5,14435.423445,2,15510.576882,4.027158,26,0.222558,0.266786,0.515078,2405.903907
1,48,Male,Self-Employed,26174.922827,32,2635.203357,1,985.607164,1,580.287785,...,5,11263.099341,2,14819.436498,2.323748,12,1.695558,0.566148,0.996432,1877.183223
2,34,Other,Unemployed,75566.837265,14,2334.341061,9,994.310119,42,564.013508,...,3,15017.144132,5,10909.806507,5.031703,47,1.759799,0.144371,0.343095,3754.286033
3,62,Male,Self-Employed,35197.961516,179,2425.384332,10,366.115346,4,838.489200,...,1,12626.138476,2,500.000000,2.787485,45,0.436117,0.014205,0.372913,6313.069238
4,27,Female,Self-Employed,12136.998349,225,10.000000,8,786.752258,16,462.495522,...,1,13576.704320,4,9514.591618,0.893892,54,1.697432,0.783868,1.902397,6788.352160


## Train and compare models

In [5]:
results = run_experiment(data_path)
results_table(results).round(4)

,model,test_accuracy,test_roc_auc,test_precision,test_recall,test_f1,cv_accuracy_mean,cv_roc_auc_mean,cv_precision_mean,cv_recall_mean,cv_f1_mean,cv_accuracy_std,cv_roc_auc_std,cv_precision_std,cv_recall_std,cv_f1_std
0,Logistic Regression,0.716,0.8063,0.8964,0.7115,0.7933,0.7510,0.8278,0.9202,0.7388,0.8196,0.0157,0.0146,0.0086,0.0150,0.0124
1,Random Forest,0.718,0.8299,0.9859,0.6410,0.7769,0.7345,0.8223,0.9847,0.6637,0.7928,0.0094,0.0098,0.0068,0.0157,0.0097


## Feature importance

In [6]:
for result in results:
    display(result.name)
    display(result.feature_importance.head(15).round(4))

'Logistic Regression'

,feature,coefficient,abs_value
0,late_payment_count,1.8612,1.8612
1,gender_Female,0.2254,0.2254
2,employment_status_Unemployed,0.1705,0.1705
3,employment_status_Employed,0.1437,0.1437
4,gender_Male,0.1205,0.1205
5,loan_application_amount,0.1145,0.1145
6,employment_status_Self-Employed,0.0935,0.0935
7,num_deposits_per_month,0.0674,0.0674
8,debit_card_usage_frequency,-0.0653,0.0653
9,gender_Other,0.0618,0.0618


'Random Forest'

,feature,importance,abs_value
0,late_payment_count,0.6513,0.6513
1,avg_monthly_balance,0.0210,0.0210
2,annual_income,0.0209,0.0209
3,loan_to_income_ratio,0.0201,0.0201
4,debit_card_spending,0.0200,0.0200
5,total_credit_pressure,0.0197,0.0197
6,loan_application_amount,0.0193,0.0193
7,account_age_months,0.0192,0.0192
8,income_to_debt_ratio,0.0191,0.0191
9,total_outstanding_debt,0.0190,0.0190


## Proxy and segment audit

This checks whether `late_payment_count` is carrying too much of the model signal and reports held-out segment metrics by `gender` and `employment_status`.

In [7]:
audit = run_audit(data_path)

display(audit.proxy_summary.round(4))
display(audit.proxy_by_value.round(4))

,feature,single_feature_roc_auc,spearman_corr,min_group_high_risk_rate,max_group_high_risk_rate,risk_rate_spread,proxy_strength
0,late_payment_count,0.827,0.4835,0.5201,1.0,0.4799,high


,late_payment_count,n,high_risk_rate
0,0,660,0.5348
1,1,598,0.5201
2,2,646,0.5325
3,3,609,0.5484
4,4,636,1.0000
5,5,630,1.0000
6,6,611,1.0000
7,7,610,1.0000


In [8]:
audit.feature_drop_comparison[[
    "scenario",
    "model",
    "test_accuracy",
    "test_roc_auc",
    "test_precision",
    "test_recall",
    "test_f1",
    "delta_test_roc_auc",
    "delta_test_recall",
    "delta_test_f1",
]].round(4)

,scenario,model,test_accuracy,test_roc_auc,test_precision,test_recall,test_f1,delta_test_roc_auc,delta_test_recall,delta_test_f1
0,with_late_payment_count,Logistic Regression,0.716,0.8063,0.8964,0.7115,0.7933,0.000,0.0000,0.0000
1,with_late_payment_count,Random Forest,0.718,0.8299,0.9859,0.6410,0.7769,0.000,0.0000,0.0000
2,without_late_payment_count,Logistic Regression,0.500,0.4863,0.7628,0.5039,0.6069,-0.320,-0.2076,-0.1864
3,without_late_payment_count,Random Forest,0.749,0.4918,0.7663,0.9674,0.8552,-0.338,0.3264,0.0783


In [9]:
audit.segment_metrics.round(4)

,model,segment_feature,segment_value,n,actual_high_risk_rate,predicted_high_risk_rate,mean_predicted_probability,accuracy,precision,recall,f1,false_positive_rate,false_negative_rate
0,Logistic Regression,employment_status,Employed,361,0.7645,0.6094,0.5963,0.6842,0.8682,0.6920,0.7702,0.3412,0.3080
1,Logistic Regression,employment_status,Self-Employed,305,0.7541,0.6197,0.5981,0.7475,0.9048,0.7435,0.8162,0.2400,0.2565
2,Logistic Regression,employment_status,Unemployed,334,0.7784,0.5958,0.5932,0.7216,0.9196,0.7038,0.7974,0.2162,0.2962
3,Logistic Regression,gender,Female,352,0.7301,0.5824,0.5876,0.7102,0.8780,0.7004,0.7792,0.2632,0.2996
4,Logistic Regression,gender,Male,314,0.7771,0.6465,0.6167,0.7166,0.8818,0.7336,0.8009,0.3429,0.2664
5,Logistic Regression,gender,Other,334,0.7934,0.5988,0.5849,0.7216,0.9300,0.7019,0.8000,0.2029,0.2981


In [10]:
results_table(results)[[
    "model",
    "test_roc_auc",
    "cv_roc_auc_mean",
    "cv_roc_auc_std",
]].round(4)

,model,test_roc_auc,cv_roc_auc_mean,cv_roc_auc_std
0,Logistic Regression,0.8063,0.8278,0.0146
1,Random Forest,0.8299,0.8223,0.0098
